In [ ]:
# Install necessary libraries
!apt-get install -y poppler-utils

!pip install -q pdfplumber pytesseract sentence-transformers chromadb pypdf Pillow pdf2image opencv-python transformers pandas

# Install the latest versions properly
!pip install -q --upgrade langchain chromadb sentence-transformers transformers accelerate bitsandbytes

# 🛠️ Install additional compatibility packages
!pip install -q langchain-community langchain-core



Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 34 not upgraded.
Need to get 186 kB of archives.
After this operation, 696 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.7 [186 kB]
Fetched 186 kB in 1s (307 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 126101 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.7_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.7) ...
Setting up poppler-utils (22.02.0-2ubuntu0.7) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Import necessary libraries
import pdfplumber
import pytesseract
import cv2
import os
import torch
import pandas as pd
from PIL import Image
from pdf2image import convert_from_path
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
from chromadb.config import Settings
from google.colab import drive


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define paths to your PDFs
pdf_paths = [
    '/content/drive/MyDrive/Data/FYP-Handbook.pdf',
    '/content/drive/MyDrive/Data/Financials.pdf',
    '/content/drive/MyDrive/Data/Annual Report.pdf'
]

# Initialize device and models
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"[INFO] Using device: {device}")

# Load models
text_encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2').to(device)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


[INFO] Using device: cuda


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
import pdfplumber
import nltk
from nltk.tokenize import sent_tokenize
import pytesseract
from pdf2image import convert_from_path
import cv2
from PIL import Image

# Download NLTK sentence tokenizer if needed
nltk.download('punkt')
nltk.download('punkt_tab')

# Helper function to extract small text chunks from a PDF
def extract_text_from_pdf(pdf_path, chunk_size=400, overlap=50):
    """Extracts text from a PDF and splits it into overlapping chunks."""
    text_chunks = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if text:
                sentences = sent_tokenize(text)
                current_chunk = ""
                for sentence in sentences:
                    if len(current_chunk) + len(sentence) <= chunk_size:
                        current_chunk += " " + sentence
                    else:
                        text_chunks.append({'type': 'text', 'content': current_chunk.strip(), 'page': i})

                        # Start new chunk with overlap
                        if overlap > 0:
                            # Take last `overlap` characters from current chunk as start of next
                            current_chunk = current_chunk[-overlap:] + " " + sentence
                        else:
                            current_chunk = sentence
                if current_chunk:
                    text_chunks.append({'type': 'text', 'content': current_chunk.strip(), 'page': i})
    return text_chunks

# Helper function to extract images from a PDF
def extract_images_from_pdf(pdf_path):
    """Extracts images from a PDF."""
    images = []
    pages = convert_from_path(pdf_path, dpi=300)
    for idx, page in enumerate(pages):
        img_path = f'/content/page_{idx}.png'
        page.save(img_path, 'PNG')
        images.append({'path': img_path, 'page': idx})
    return images

# Helper function to perform OCR on an image
def perform_ocr_on_image(image_path):
    """Performs OCR on an image."""
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    text = pytesseract.image_to_string(img_rgb)
    return text

# Helper function to get text embeddings (using Sentence-BERT)
def get_text_embedding(text):
    """Generates text embeddings using Sentence-BERT."""
    emb = text_encoder.encode(text, convert_to_tensor=True)
    return emb.detach().cpu().numpy()

# Helper function to get image embeddings (using CLIP)
def get_image_embedding(image_path):
    """Generates image embeddings using CLIP."""
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    outputs = clip_model.get_image_features(**inputs)
    outputs = outputs / outputs.norm(dim=-1, keepdim=True)
    return outputs.squeeze(0).detach().cpu().numpy()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# Extraction Phase
all_data = []

for pdf_path in pdf_paths:
    print(f"\n========== Starting processing for {pdf_path} ==========")

    # Extract text and images
    text_chunks = extract_text_from_pdf(pdf_path)
    images_info = extract_images_from_pdf(pdf_path)

    # Add text chunks to the data
    for chunk in text_chunks:
        all_data.append({
            'type': 'text',
            'content': chunk['content'],
            'image_path': None,
            'ocr_text': None,
            'page': chunk['page']
        })

    # Add image info to the data
    for img_info in images_info:
        ocr_text = perform_ocr_on_image(img_info['path'])
        all_data.append({
            'type': 'image',
            'content': None,
            'image_path': img_info['path'],
            'ocr_text': ocr_text,
            'page': img_info['page']
        })

# Convert data into a DataFrame
df = pd.DataFrame(all_data)

# Save the DataFrame
csv_save_path = '/content/drive/MyDrive/Data/extracted_data.csv'
df.to_csv(csv_save_path, index=False, encoding='utf-8-sig')
print(f"[SUCCESS] Data saved at: {csv_save_path}")



========== Starting processing for /content/drive/MyDrive/Data/FYP-Handbook.pdf ==========

========== Starting processing for /content/drive/MyDrive/Data/Financials.pdf ==========



========== Starting processing for /content/drive/MyDrive/Data/Annual Report.pdf ==========


[SUCCESS] Data saved at: /content/drive/MyDrive/Data/extracted_data.csv


In [ ]:
df

,type,content,image_path,ocr_text,page
0,text,BS FINAL YEAR PROJECT\nHANDBOOK 2023\n(FAST SC...,None,None,0
1,text,BS FINAL YEAR PROJECT\nHANDBOOK 2023\n(FAST SC...,None,None,1
2,text,BS Final Year Project Handbook 2023\n02\n03\n0...,None,None,3
3,text,BS Final Year Project Handbook 2023\n02 FAST-N...,None,None,4
4,text,BS Final Year Project Handbook 2023\nOctober\n...,None,None,5
...,...,...,...,...,...
1035,image,None,/content/page_80.png,5. Job Fair 2024\n\nThe Job Fair/Open House 20...,80
1036,image,None,/content/page_81.png,"UpStart Commerce Signed on September 12, 2023,...",81
1037,image,None,/content/page_82.png,11.Outreach Activities\n\no Facilitated the vi...,82
1038,image,None,/content/page_83.png,Chapter-9 Health Care Facilities\n\nFAST-NUCES...,83


In [ ]:
# Embedding Phase
print("\n========== Starting Embedding Phase ==========")
embeddings = []
embedding_types = []  # Track whether it's text or image

# Process each row of the DataFrame
for idx, row in df.iterrows():
    if row['type'] == 'text':
        emb = get_text_embedding(row['content'])
        embedding_types.append('text')
    elif row['type'] == 'image':
        emb = get_image_embedding(row['image_path'])
        embedding_types.append('image')
    embeddings.append(emb)

    if idx % 10 == 0:
        print(f"[INFO] Embedded {idx} chunks...")

# Separate embeddings for text and image
text_embeddings = []
image_embeddings = []

for i, emb_type in enumerate(embedding_types):
    if emb_type == 'text':
        text_embeddings.append(embeddings[i])
    elif emb_type == 'image':
        image_embeddings.append(embeddings[i])

# Convert to numpy arrays for easy processing
text_embeddings = np.vstack(text_embeddings)
image_embeddings = np.vstack(image_embeddings)

print(f"[INFO] Text embeddings shape: {text_embeddings.shape}")
print(f"[INFO] Image embeddings shape: {image_embeddings.shape}")

# Save embeddings
text_embeddings_df = pd.DataFrame(text_embeddings)
text_embeddings_df.to_csv('/content/drive/MyDrive/Data/text_embeddings.csv', index=False, encoding='utf-8-sig')
print("[SUCCESS] Text embeddings saved.")

image_embeddings_df = pd.DataFrame(image_embeddings)
image_embeddings_df.to_csv('/content/drive/MyDrive/Data/image_embeddings.csv', index=False, encoding='utf-8-sig')
print("[SUCCESS] Image embeddings saved.")



========== Starting Embedding Phase ==========
[INFO] Embedded 0 chunks...
[INFO] Embedded 10 chunks...
[INFO] Embedded 20 chunks...
[INFO] Embedded 30 chunks...
[INFO] Embedded 40 chunks...
[INFO] Embedded 50 chunks...
[INFO] Embedded 60 chunks...
[INFO] Embedded 70 chunks...
[INFO] Embedded 80 chunks...
[INFO] Embedded 90 chunks...
[INFO] Embedded 100 chunks...
[INFO] Embedded 110 chunks...
[INFO] Embedded 120 chunks...
[INFO] Embedded 130 chunks...
[INFO] Embedded 140 chunks...
[INFO] Embedded 150 chunks...
[INFO] Embedded 160 chunks...
[INFO] Embedded 170 chunks...
[INFO] Embedded 180 chunks...
[INFO] Embedded 190 chunks...
[INFO] Embedded 200 chunks...
[INFO] Embedded 210 chunks...
[INFO] Embedded 220 chunks...
[INFO] Embedded 230 chunks...
[INFO] Embedded 240 chunks...
[INFO] Embedded 250 chunks...
[INFO] Embedded 260 chunks...
[INFO] Embedded 270 chunks...
[INFO] Embedded 280 chunks...
[INFO] Embedded 290 chunks...
[INFO] Embedded 300 chunks...
[INFO] Embedded 310 chunks...
[IN

# **DATABASE SETUP (1)**

In [ ]:
# Initialize ChromaDB
print("\n[INFO] Initializing ChromaDB...")
chroma_client = chromadb.PersistentClient(path="/content/chroma_db")

# Separate DataFrames for text and image
text_df = df[df['type'] == 'text'].reset_index(drop=True)
image_df = df[df['type'] == 'image'].reset_index(drop=True)

# --------------------------- #
# Insert Text Embeddings
# --------------------------- #
print("\n[INFO] Creating collection for TEXT data...")
text_collection = chroma_client.create_collection(name="pdf_text_data", get_or_create=True)

text_ids = [f"text_{i}" for i in range(len(text_df))]
text_metadatas = text_df.to_dict(orient='records')

# Handle None values in metadata
text_metadatas = [{k: (v if v is not None else '') for k, v in meta.items()} for meta in text_metadatas]

text_documents = text_df['content'].tolist()

text_collection.add(
    embeddings=text_embeddings.tolist(),
    documents=text_documents,
    metadatas=text_metadatas,
    ids=text_ids
)
print("[SUCCESS] Text data inserted into ChromaDB collection 'pdf_text_data'.")

# --------------------------- #
# Insert Image Embeddings
# --------------------------- #
print("\n[INFO] Creating collection for IMAGE data...")
image_collection = chroma_client.create_collection(name="pdf_image_data", get_or_create=True)

image_ids = [f"image_{i}" for i in range(len(image_df))]
image_metadatas = image_df.to_dict(orient='records')

# Handle None values in metadata
image_metadatas = [{k: (v if v is not None else '') for k, v in meta.items()} for meta in image_metadatas]

image_documents = image_df['ocr_text'].tolist()

image_collection.add(
    embeddings=image_embeddings.tolist(),
    documents=image_documents,
    metadatas=image_metadatas,
    ids=image_ids
)
print("[SUCCESS] Image data inserted into ChromaDB collection 'pdf_image_data'.")

# Persist the database
print("\n✅ ChromaDB persisted automatically at '/content/chroma_db'!")



[INFO] Initializing ChromaDB...

[INFO] Creating collection for TEXT data...
[SUCCESS] Text data inserted into ChromaDB collection 'pdf_text_data'.

[INFO] Creating collection for IMAGE data...
[SUCCESS] Image data inserted into ChromaDB collection 'pdf_image_data'.

✅ ChromaDB persisted automatically at '/content/chroma_db'!


In [ ]:
# Fetch and display collections from ChromaDB
print("\n[INFO] Fetching data from 'pdf_text_data' collection...")
text_results = text_collection.get(include=["documents", "metadatas", "embeddings"])

# Flatten and display text data
text_documents = [doc for doc in text_results['documents']]
text_metadatas = [meta for meta in text_results['metadatas']]
text_embeddings = [embed for embed in text_results['embeddings']]

text_df_display = pd.DataFrame({
    "Document": text_documents,
    "Metadata": text_metadatas,
    "Embeddings": text_embeddings
})

print("\nText Data from 'pdf_text_data' Collection:")
display(text_df_display)

print("\n[INFO] Fetching data from 'pdf_image_data' collection...")
image_results = image_collection.get(include=["documents", "metadatas", "embeddings"])

# Flatten and display image data
image_documents = [doc for doc in image_results['documents']]
image_metadatas = [meta for meta in image_results['metadatas']]
image_embeddings = [embed for embed in image_results['embeddings']]

image_df_display = pd.DataFrame({
    "Document": image_documents,
    "Metadata": image_metadatas,
    "Embeddings": image_embeddings
})

print("\nImage Data from 'pdf_image_data' Collection:")
display(image_df_display)


[INFO] Fetching data from 'pdf_text_data' collection...

Text Data from 'pdf_text_data' Collection:


,Document,Metadata,Embeddings
0,,"{'page': 0, 'type': 'text', 'image_path': '', ...","[-0.11883842945098877, 0.04829873517155647, -0..."
1,BS FINAL YEAR PROJECT\nHANDBOOK 2023\n(FAST SC...,"{'page': 0, 'ocr_text': '', 'type': 'text', 'i...","[-0.06644495576620102, 0.02246316336095333, -0..."
2,,"{'image_path': '', 'type': 'text', 'content': ...","[-0.11883842945098877, 0.04829873517155647, -0..."
3,BS FINAL YEAR PROJECT\nHANDBOOK 2023\n(FAST SC...,{'content': 'BS FINAL YEAR PROJECT HANDBOOK 20...,"[-0.06389405578374863, 0.03432097285985947, -0..."
4,,"{'image_path': '', 'page': 3, 'type': 'text', ...","[-0.11883842945098877, 0.04829873517155647, -0..."
...,...,...,...
1700,The campus is in close of proximity of 13 gene...,{'content': 'The campus is in close of proximi...,"[0.14886604249477386, -0.04678962007164955, 0...."
1701,One university vehicle and on-campus\nresident...,"{'ocr_text': '', 'page': 84, 'content': 'One u...","[0.01839580573141575, -0.012393934652209282, 0..."
1702,An in-house health care facility is not yet po...,"{'type': 'text', 'image_path': '', 'ocr_text':...","[0.046621911227703094, 0.004472685512155294, 0..."
1703,"For this,\nconsiderations have been made for i...","{'page': 84, 'content': 'For this, considerati...","[-0.010160948149859905, -0.0014220667071640491..."



[INFO] Fetching data from 'pdf_image_data' collection...

Image Data from 'pdf_image_data' Collection:


,Document,Metadata,Embeddings
0,= i‘ NATIONAL UNIVERSITY\n“CZ | of Computer & ...,"{'image_path': '/content/page_0.png', 'page': ...","[0.003925670869648457, -0.002987332409247756, ..."
1,NATIONAL UNIVERSITY\n\nof Computer and Emergin...,"{'content': '', 'type': 'image', 'page': 1, 'i...","[-0.030893219634890556, 0.07224931567907333, -..."
2,,"{'type': 'image', 'page': 2, 'image_path': '/c...","[-0.016384312883019447, 0.014744909480214119, ..."
3,BS Final Year Project Handbook 2023\n\nTable o...,"{'content': '', 'image_path': '/content/page_3...","[-0.01874472200870514, 0.014965505339205265, -..."
4,BS Final Year Project Handbook 2023\n\nPreface...,"{'content': '', 'type': 'image', 'ocr_text': '...","[-0.03165971860289574, 0.0074755349196493626, ..."
...,...,...,...
177,5. Job Fair 2024\n\nThe Job Fair/Open House 20...,"{'content': '', 'image_path': '/content/page_8...","[-0.011715828441083431, 0.02109253965318203, -..."
178,"UpStart Commerce Signed on September 12, 2023,...","{'content': '', 'page': 81, 'type': 'image', '...","[0.020579086616635323, 0.0056813061237335205, ..."
179,11.Outreach Activities\n\no Facilitated the vi...,"{'type': 'image', 'image_path': '/content/page...","[-0.006994336377829313, 0.016129886731505394, ..."
180,Chapter-9 Health Care Facilities\n\nFAST-NUCES...,"{'page': 83, 'type': 'image', 'ocr_text': 'Cha...","[-0.011471429839730263, 0.010425086133182049, ..."


In [ ]:
# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
# from sentence_transformers import SentenceTransformer
# import chromadb
# import pytesseract
# from PIL import Image
# import gradio as gr
# import numpy as np

# # Load a smaller model for text generation (DistilGPT-2)
# model_name = "distilgpt2"  # Lightweight GPT-2 version
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name)
# llm = pipeline("text-generation", model=model, tokenizer=tokenizer)

# # Set up the Sentence-Transformers model for generating embeddings
# embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# # Set up ChromaDB client and connect to your collection
# chroma_client = chromadb.PersistentClient(path="/content/chroma_db")  # Use the same path as before
# text_collection = chroma_client.get_collection("pdf_text_data")  # Collection holding text embeddings



In [ ]:
# # Function to retrieve relevant documents using the embedding model
# def retrieve_relevant_documents(query, top_k=3):
#     query_embedding = embedding_model.encode(query)  # Get query embedding
#     results = text_collection.query(query_embedding, n_results=top_k)  # Query the database for similar documents
#     return results['documents']  # Return the relevant documents

# # Function to generate response using the retrieved documents as context
# def generate_rag_answer(query):
#     # Retrieve relevant documents from the vector database
#     relevant_documents = retrieve_relevant_documents(query)

#     # Combine the query and retrieved documents as context
#     context = "\n".join(relevant_documents)

#     # Prepare the full prompt for the LLM (Query + Context)
#     full_prompt = f"Query: {query}\nContext: {context}\nAnswer:"

#     # Generate the response using LLM
#     response = llm(full_prompt, max_length=200, temperature=0.7)
#     return response[0]['generated_text']

# # Function to extract text from an image using OCR
# def extract_text_from_image(image):
#     text = pytesseract.image_to_string(image)
#     return text

# # Function to handle both text and image input, integrating OCR and RAG
# def handle_input(query, image=None):
#     # If an image is uploaded, extract text using OCR
#     if image:
#         image_text = extract_text_from_image(image)
#         query = f"{query} based on the image content: {image_text}"

#     # Generate the response using RAG approach
#     response = generate_rag_answer(query)

#     return response



In [ ]:
# # Gradio interface setup
# iface = gr.Interface(
#     fn=handle_input,
#     inputs=[
#         gr.Textbox(label="Enter your query", placeholder="Ask anything about the PDF..."),
#         gr.Image(type="pil", label="Upload an image (optional)")
#     ],
#     outputs="text",
#     live=True,
#     title="AI-powered PDF Assistant",
#     description="Ask questions about the content of the PDF, or upload an image for analysis."
# )

# # Launch the interface
# iface.launch()

In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 135.9 MB/s eta 0:00:00


# new test

In [ ]:
# Basic Imports
import chromadb
from langchain.vectorstores import Chroma  # Corrected import for Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import gradio as gr

In [ ]:
# Load lightweight embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Connect to ChromaDB
chroma_client = chromadb.PersistentClient(path="/content/chroma_db")
text_vectorstore = Chroma(
    client=chroma_client,
    collection_name="pdf_text_data",
    embedding_function=embedding_model,
)

<ipython-input-6-82ed5afdfd99>:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

<ipython-input-6-82ed5afdfd99>:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  text_vectorstore = Chroma(


In [ ]:
# Initialize the transformer model and tokenizer
model_name = "google/flan-t5-base"  # Lightweight but powerful for QA
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Create the HuggingFace pipeline
llm_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.2,
    repetition_penalty=1.1
)

# Wrap the pipeline to make it compatible with langchain's HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=llm_pipeline)

print("✅ Loaded FLAN-T5 model. Ready for smart AF answers!")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu


✅ Loaded FLAN-T5 model. Ready for smart AF answers!


<ipython-input-7-6498d780d21a>:17: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=llm_pipeline)


In [ ]:
# Setup Retriever (Chroma vector store)
retriever = text_vectorstore.as_retriever(search_kwargs={"k": 4})

# Custom Prompt Template
template = """
You are a helpful assistant. Use the following extracted parts of a document to answer the question at the end.
If you don't know the answer, just say "I don't know".

{context}

Question: {question}
Helpful Answer:
"""
prompt = PromptTemplate(template=template, input_variables=["context", "question"])

# Build the RetrievalQA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)

In [ ]:
# Function to answer user questions
def answer_question(user_query):
    result = qa_chain(user_query)
    answer = result['result']

    # Extract Sources (Page numbers)
    sources = []
    for doc in result['source_documents']:
        meta = doc.metadata
        page = meta.get('page', 'Unknown Page')
        sources.append(f"Page {page}")

    sources_text = "\n".join(sources) if sources else "No sources found."

    # Return nicely formatted response
    full_response = f"**Answer:**\n{answer}\n\n**Sources:**\n{sources_text}"
    return full_response

In [ ]:
# Gradio Interface
iface = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(lines=2, placeholder="Ask me anything about your university FAST NUCES..."),
    outputs="markdown",
    title="📚 Smart FYP Assistant",
    description="Ask any question related to your PDF data. Powered by RAG 🔥",
    theme="soft",
    css="body {background: linear-gradient(to bottom right, #E0EAFC, #CFDEF3);} textarea {font-size: 18px;} .output_markdown {font-size: 18px;}"
)

# Launch the App
iface.launch(debug=True)


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3578536e2d1904115d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


<ipython-input-9-98984463a9c3>:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain(user_query)
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `